In [1]:
#DQN Hyperparameter tuning
from Singe_ag_Environment_SparseR import SingleSatelliteEnvSR
from gymnasium.utils.env_checker import check_env
import traceback
print(type(SingleSatelliteEnvSR))
# This will catch many common issues
env = SingleSatelliteEnvSR("rgb_array")
try:
    check_env(env)
    print("Environment passes all checks!")
except Exception as e:
    print("Environment has issues:")
    traceback.print_exc()

<class 'type'>
[0.7  0.3  0.   0.9  0.6  0.5  0.5  0.4  0.5  0.6  0.85 1.  ]
Environment passes all checks!


/home/ethan/Documents/Research/Single_Agent_EO_Project/CleanAgvenv/lib/python3.10/site-packages/gymnasium/utils/env_checker.py:321: UserWarning: WARN: Not able to test alternative render modes due to the environment not having a spec. Try instantialising the environment through gymnasium.make
  logger.warn(


In [2]:
import optuna

from cleanrl_utils.tuner import Tuner

dqntuner = Tuner(
    script="cleanrl/dqn.py",
    metric="charts/episodic_return",
    metric_last_n_average_window=50,
    direction="maximize",
    aggregation_type="average",
    target_scores={
        "SingleSatelliteEnvSR":None
    },
    params_fn=lambda trial: {
        "learning-rate": trial.suggest_float("learning-rate", 0.0003, 0.003, log=True),
        "buffer_size": trial.suggest_categorical("buffer_size", [10000, 100000, 1000000]),
        "tau": trial.suggest_float("tau",0,1),
        "target_network_frequency": trial.suggest_categorical("target_network_frequency", [500,1000, 1500,]),
        "batch_size": trial.suggest_categorical("batch_size", [16, 32,64]),
        "start_e": trial.suggest_float("start_e", 0.5,1),
        "end_e":trial.suggest_float("end_e", 0,0.5),
        "exploration_fraction":trial.suggest_float("end_e", 0,0.5),
        "learning_starts":trial.suggest_int("learning_starts", 10000,1000000),
        "train_frequency":trial.suggest_int("train_frequency",1,10),
        "total-timesteps": 500000,
        "num-envs": 1,
    },
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
    sampler=optuna.samplers.TPESampler(),
)
dqntuner.tune(
    num_trials=100,
    num_seeds=5,
)

/home/ethan/Documents/Research/Single_Agent_EO_Project/CleanAgvenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-12-12 19:52:09,248] A new study created in RDB with name: tuner_1765587128


==========================================================================================

run another tuner with the following command:

python -m cleanrl_utils.tuner --study-name tuner_1765587128

==========================================================================================

[W 2025-12-12 19:52:09,354] Trial 0 failed with parameters: {'learning-rate': 0.0006765639911412136, 'buffer_size': 1000000, 'tau': 0.21503968822868358, 'target_network_frequency': 1500, 'batch_size': 16, 'start_e': 0.7079372667265369, 'end_e': 0.2560651490310348, 'learning_starts': 783713, 'train_frequency': 5} because of the following error: FileNotFoundError(2, 'No such file or directory').
Traceback (most recent call last):
  File "/home/ethan/Documents/Research/Single_Agent_EO_Project/CleanAgvenv/lib/python3.10/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "/home/ethan/Documents/Research/Single_Agent_EO_Project/cleanrl/cleanrl_utils/tuner.py", line 92, in objective
    experiment = runpy.run_path(path_name=self.script, run_name="__main__")
  File "/usr/lib/python3.10/runpy.py", line 288, in run_path
    code, fname = _get_code_from_file(run_name, path_name)
  File "/usr/lib/python3.10/runpy.py", line 252, in _get_code_fr

FileNotFoundError: [Errno 2] No such file or directory: '/home/ethan/Documents/Research/Single_Agent_EO_Project/cleanrl/dqn.py'